In [8]:
import re

def remove_prefix_from_content(chat, prefixes=("Observation: ", "Feedback: ")):
    chat = chat.copy()
    if 'content' in chat:
        content = chat['content']
        for prefix in prefixes:
            if content.startswith(prefix):
                content = content[len(prefix):]
        chat['content'] = content
    return chat

new_chats_raw = data['messages'][2:-1]

# Remove "Observation: " and "Feedback: " from contents
new_chats = [remove_prefix_from_content(chat) for chat in new_chats_raw]

num_experiences = len(new_chats)//3

experiences = []
for i in range(num_experiences):
    experiences.append(new_chats[3*i:3*i+3])

experiences[0]


KeyError: 'messages'

In [ ]:
import json
# Load the experiences from a JSONL file
experiences = []
with open("experiences.jsonl", "r") as fin:
    for line in fin:
        experiences.append(json.loads(line))


In [ ]:
print(new_dataset[0]['input_messages'][2:10])

[{'role': 'assistant', 'content': '\\boxed{Lunar Sol}'}, {'role': 'user', 'content': 'Correct! The answer was \\boxed{Lunar Sol}.'}, {'role': 'user', 'content': "What is the name of the city with the following description: The weather is rainy, with ongoing precipitation that clearly marks the day as rainy. The city's economy centers on agriculture, with expansive farmlands and seasonal harvests. Conditions show desert geography, with dry climate and limited plant life."}, {'role': 'assistant', 'content': '\\boxed{Grand Evergreen}'}, {'role': 'user', 'content': 'Your answer is incorrect. The correct answer was \\boxed{Grand Sol}.'}, {'role': 'user', 'content': 'What is the name of the city with the following description: Terrain classification: mountains, marked by high elevation and steep gradients. Supply chains and skilled trades form the backbone of this manufacturing hub. The weather is sunny, with clear skies and bright daylight; conditions are explicitly categorized as sunny.'},

## Mixed dataset

In [9]:
import json
import random
from tqdm import tqdm
from collections import Counter
import os

# Paths to datasets
dataset1_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_non_collapsed_boxed_only_20000_with_subsample_and_original_experiences/dataset_with_up_to_50_new_cities_distractors.jsonl"
dataset2_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_question_gen-20_from_100_val_data_answer_gen-500_train_student_messages_none_2500/dataset.jsonl"
dataset3_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_question_gen-1to50_from_100_val_data_answer_gen-same_as_question_gen_shuffled_student_messages_teacher_context_2500/dataset.jsonl"

# Helper function to load dataset and filter by output_ids presence
def load_and_filter_jsonl(path):
    total = 0
    filtered = 0
    valid_entries = []
    with open(path, "r") as f:
        for line in tqdm(f, desc=f"Loading {path.split('/')[-1]}"):
            total += 1
            try:
                entry = json.loads(line)
            except Exception as e:
                print(f"Corrupt line detected: {e}")
                continue
            output_ids = entry.get("output_ids", None)
            # We treat null/None output_ids as needing to be filtered
            if output_ids is not None:
                valid_entries.append(entry)
            else:
                filtered += 1
    print(f"\nStats for {path}:")
    print(f"  Total entries: {total}")
    print(f"  Filtered out (output_ids is None): {filtered}")
    print(f"  Valid (kept): {len(valid_entries)}")
    return valid_entries, total, filtered

# # Load and filter each dataset
# ds1, tot1, filt1 = load_and_filter_jsonl(dataset1_path)
# ds2, tot2, filt2 = load_and_filter_jsonl(dataset2_path)
# ds3, tot3, filt3 = load_and_filter_jsonl(dataset3_path)

# # For dataset1, select 15000 random entries (after filtering)
# random.seed(42)
# if len(ds1) < 15000:
#     print(f"WARNING: Requested 15000 from ds1, but only {len(ds1)} valid entries available. Using all of them.")
#     ds1_sample = ds1
# else:
#     ds1_sample = random.sample(ds1, 15000)

# # Use all of ds2 and ds3
# mixed_dataset = ds1_sample + ds2 + ds3
# random.seed(1337)
# random.shuffle(mixed_dataset)

# print("\n--- Dataset Statistics ---")
# print(f"Sampled from ds1 (with output_ids): {len(ds1_sample)}")
# print(f"Added all from ds2 (with output_ids): {len(ds2)}")
# print(f"Added all from ds3 (with output_ids): {len(ds3)}")
# print(f"Final mixed dataset size: {len(mixed_dataset)}")
# print(f"Total rows (before filtering) in ds1: {tot1}, filtered: {filt1}")
# print(f"Total rows (before filtering) in ds2: {tot2}, filtered: {filt2}")
# print(f"Total rows (before filtering) in ds3: {tot3}, filtered: {filt3}")

# # Optionally, print some stats on output_ids distribution
# output_ids_lengths = [len(entry["output_ids"]) if "output_ids" in entry and entry["output_ids"] is not None else 0 for entry in mixed_dataset]
# length_counter = Counter(output_ids_lengths)
# print("\nDistribution of output_ids lengths in mixed dataset (length:count):")
# for l, cnt in sorted(length_counter.items()):
#     print(f"  {l}: {cnt}")

# # Save the mixed dataset to a new file
# mixed_save_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_mixed_15000_boxed_20000_distractors_plus_two_val_datasets/dataset.jsonl"
# mixed_save_dir = os.path.dirname(mixed_save_path)
# os.makedirs(mixed_save_dir, exist_ok=True)  # Ensure the directory exists

# with open(mixed_save_path, "w") as fout:
#     for entry in tqdm(mixed_dataset, desc="Saving mixed dataset"):
#         fout.write(json.dumps(entry) + "\n")

# print(f"\nSaved mixed dataset to {mixed_save_path}")


In [11]:
ds, tot, filt = load_and_filter_jsonl(path="/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_second_half_seed_23_20000_with_subsample_and_original_experiences/dataset.jsonl")

Loading dataset.jsonl: 0it [00:00, ?it/s]

Loading dataset.jsonl: 20000it [00:19, 1043.53it/s]


Stats for /projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_second_half_seed_23_20000_with_subsample_and_original_experiences/dataset.jsonl:
  Total entries: 20000
  Filtered out (output_ids is None): 520
  Valid (kept): 19480


In [12]:
import json
from tqdm import tqdm

# Save the filtered dataset (ds) to the original path, overwriting
save_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_second_half_seed_23_20000_with_subsample_and_original_experiences/dataset.jsonl"

with open(save_path, "w") as fout:
    for entry in tqdm(ds, desc="Saving filtered dataset"):
        fout.write(json.dumps(entry) + "\n")

print(f"Filtered dataset saved to {save_path}")



Saving filtered dataset: 100%|██████████| 19480/19480 [00:18<00:00, 1037.82it/s]

Filtered dataset saved to /projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_second_half_seed_23_20000_with_subsample_and_original_experiences/dataset.jsonl


In [13]:
import json
from collections import Counter
from tqdm import tqdm

# Load the dataset from the filtered file
load_path = "/projects/bfsg/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_second_half_seed_23_20000_with_subsample_and_original_experiences/dataset.jsonl"

dataset = []
with open(load_path, "r") as f:
    for line in tqdm(f, desc="Loading dataset.jsonl"):
        dataset.append(json.loads(line))

print(f"\nTotal number of entries loaded: {len(dataset)}\n")

# Optionally count on a field (e.g., input, output, or any class/label if exists)
# Example: count by label if 'label' in entry
# label_counts = Counter(entry.get("label") for entry in dataset)
# print("Counts by label:")
# for k, v in label_counts.items():
#     print(f"  {k}: {v}")

# Print the first 3 entries for inspection
print("First 3 entries:")
for entry in dataset[:3]:
    print(json.dumps(entry, indent=2))
    print("-" * 60)


Loading dataset.jsonl: 19480it [00:20, 972.44it/s] 


Total number of entries loaded: 19480

First 3 entries:
{
  "metadata": {
    "generation_index": 0
  },
  "type": "memory_distillation",
  "input_messages": [
    {
      "role": "system",
      "content": "You are a continual learning assistant that maps a description of a city to its name. At every step, you will be given the description for a city and asked to identify its name. After this, you will receive feedback on if your guess is correct or incorrect. If the guess was incorrect, you will receive the correct city name. At the beginning, you will not know the correct answer for any city. As you solve more questions, you will find the correct mapping from description to city name for more cities which you can use to get the correct answer in future questions.\n\nTask:\n\n- Read the city's natural-language description (which may include its dominant industry, terrain, and weather).\n- Look through the description of cities you have seen before, and choose the city that best matc

In [4]:
import json

mixed_data_path = "/data/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_mixed_15000_boxed_20000_distractors_plus_two_val_datasets/dataset.jsonl"

mixed_items = []
with open(mixed_data_path, "r") as fin:
    for i, line in enumerate(fin):
        mixed_items.append(json.loads(line))
        if i == 10:
            break

print("First two items from mixed dataset:")
for i, item in enumerate(mixed_items):
    print(f"\n--- Item {i+1} ---")
    print(len(item['input_messages']))


First two items from mixed dataset:

--- Item 1 ---
8

--- Item 2 ---
50

--- Item 3 ---
131

--- Item 4 ---
2

--- Item 5 ---
65

--- Item 6 ---
5

--- Item 7 ---
38

--- Item 8 ---
53

--- Item 9 ---
20

--- Item 10 ---
116

--- Item 11 ---
2


## Preping ICL examples/distractors

In [1]:
import json
memories_path = "/scratch/m000122/stalaei/logs/continual_learning/outputs/stalaei_cities_easy/history_agent/easy_synth_cities_40_25_l8b_non_collapsed_only_boxed/20251103_181024/memories/memory_1000.jsonl"

with open(memories_path, "r") as fin:
    memories = [json.loads(line) for line in fin]
    memories = memories[-1500:]

print(f"Memories length: {len(memories)}")

assert len(memories) % 3 == 0, f"Memory length is not divisible by 3: {len(memories)}"

experiences = []
for i in range(len(memories) // 3):
    observation = memories[3*i]
    action = memories[3*i+1]
    feedback = memories[3*i+2]
    experiences.append(
        [
            {
                "role": "user",
                "content": observation["content"]
            },
            {
                "role": "assistant",
                "content": action["content"]
            },
            {
                "role": "user",
                "content": feedback["content"]
            }
        ]
    )
    
experiences[0], experiences[1], experiences[120]


Memories length: 1500


([{'role': 'user',
   'content': 'What is the name of the city with the following description: The terrain is coastal, characterized by proximity to the ocean and maritime features. It is rainy with persistent rainfall, and the conditions are explicitly classified as rainy. High-speed connectivity and a skilled workforce define its technology ecosystem.'},
  {'role': 'assistant', 'content': '\\boxed{Serene Vanguard}'},
  {'role': 'user',
   'content': 'Your answer is incorrect. The correct answer was \\boxed{Starlit Stone}.'}],
 [{'role': 'user',
   'content': "What is the name of the city with the following description: Today's weather is unequivocally sunny, featuring persistent sunshine and no cloud cover of note. Terrain classification: coastal, marked by ocean adjacency and maritime climate. Mining underpins the economy, with extraction sites and processing facilities nearby."},
  {'role': 'assistant', 'content': '\\boxed{Silver Cedar}'},
  {'role': 'user',
   'content': 'Your ans

In [ ]:
# import json

# experiences_path = "/scratch/m000122/stalaei/logs/continual_learning/outputs/stalaei_cities_easy/history_agent/easy_synth_cities_40_25_l8b_non_collapsed_only_boxed/20251103_181024/experiences.jsonl"
# with open(experiences_path, "w") as fout:
#     for exp in experiences:
#         fout.write(json.dumps(exp, ensure_ascii=False) + "\n")
# print(f"Saved {len(experiences)} experiences to {experiences_path}")


Saved 500 experiences to /scratch/m000122/stalaei/logs/continual_learning/outputs/stalaei_cities_easy/history_agent/easy_synth_cities_40_25_l8b_non_collapsed_only_boxed/20251103_181024/experiences.jsonl


In [4]:
import json

# Path to the saved experiences file
experiences_path = "/data/stalaei/logs/continual_learning/outputs/stalaei_cities_easy/history_agent/easy_synth_cities_40_25_l8b_non_collapsed_only_boxed/20251103_181024/experiences.jsonl"

# Load 500 experiences
loaded_experiences = []
with open(experiences_path, "r") as fin:
    for i, line in enumerate(fin):
        if i >= 500:
            break
        loaded_experiences.append(json.loads(line))

print(f"Loaded {len(loaded_experiences)} experiences from {experiences_path}")


Loaded 500 experiences from /data/stalaei/logs/continual_learning/outputs/stalaei_cities_easy/history_agent/easy_synth_cities_40_25_l8b_non_collapsed_only_boxed/20251103_181024/experiences.jsonl


## Data set of past experiences with distractors and ICL encouragement

In [1]:
import json
import random
from tqdm import tqdm
from collections import Counter
import os

# Paths to datasets
dataset1_path = "/scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_non_collapsed_boxed_only_20000_with_subsample_and_original_experiences/dataset_with_up_to_50_new_cities_distractors.jsonl"
dataset3_path = "/scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_question_gen-1to50_from_100_val_data_answer_gen-same_as_question_gen_shuffled_student_messages_teacher_context_2500/dataset.jsonl"

# Helper function to load dataset and filter by output_ids presence
def load_and_filter_jsonl(path):
    total = 0
    filtered = 0
    valid_entries = []
    with open(path, "r") as f:
        for line in tqdm(f, desc=f"Loading {path.split('/')[-1]}"):
            total += 1
            try:
                entry = json.loads(line)
            except Exception as e:
                print(f"Corrupt line detected: {e}")
                continue
            output_ids = entry.get("output_ids", None)
            # We treat null/None output_ids as needing to be filtered
            if output_ids is not None:
                valid_entries.append(entry)
            else:
                filtered += 1
    print(f"\nStats for {path}:")
    print(f"  Total entries: {total}")
    print(f"  Filtered out (output_ids is None): {filtered}")
    print(f"  Valid (kept): {len(valid_entries)}")
    return valid_entries, total, filtered

# Load and filter each dataset (only 1 and 3)
ds1, tot1, filt1 = load_and_filter_jsonl(dataset1_path)
ds3, tot3, filt3 = load_and_filter_jsonl(dataset3_path)

Loading dataset_with_up_to_50_new_cities_distractors.jsonl: 19574it [00:22, 884.50it/s] 



Stats for /scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_non_collapsed_boxed_only_20000_with_subsample_and_original_experiences/dataset_with_up_to_50_new_cities_distractors.jsonl:
  Total entries: 19574
  Filtered out (output_ids is None): 0
  Valid (kept): 19574


Loading dataset.jsonl: 2500it [00:01, 1972.40it/s]


Stats for /scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_gen_l8b_chatboxed_question_gen-1to50_from_100_val_data_answer_gen-same_as_question_gen_shuffled_student_messages_teacher_context_2500/dataset.jsonl:
  Total entries: 2500
  Filtered out (output_ids is None): 2
  Valid (kept): 2498


In [2]:
len(ds1[9]['input_messages'])

17

In [3]:
import random
from tqdm import tqdm
from copy import deepcopy

def take_k_random_first_experiences(experiences, min_experiences, max_experiences):
    num_experiences = random.randint(min_experiences, max_experiences)
    chosen_experiences = experiences[:num_experiences]
    chats = []
    for exp in chosen_experiences:
        chats.extend(exp)
    return chats
    
def take_k_random_experiences(experiences, min_experiences, max_experiences):
    num_experiences = random.randint(min_experiences, max_experiences)
    chosen_experiences = random.sample(experiences, num_experiences)
    chats = []
    for exp in chosen_experiences:
        chats.extend(exp)
    return chats

num_repeats = 1

new_ds1 = []

for repeat in tqdm(range(num_repeats)):
    for i in tqdm(range(len(ds1))):
        row = deepcopy(ds1[i])
        row['input_messages'] = [
            ds1[i]['input_messages'][0],
            # *take_k_random_experiences(ds1, 0, 50),
            ds1[i]['input_messages'][-1]
        ]
        new_ds1.append(row)


100%|██████████| 1/1 [00:40<00:00, 40.94s/it]


In [5]:
len(new_ds1[11]['input_messages']), len(new_ds1)

(2, 19574)

In [ ]:
# Flexible unified dataset creation from multiple splits

def unified_sample_and_mix(splits_and_counts, shuffle_seed=1337, individual_seeds=None, print_stats=True):
    """
    Args:
        splits_and_counts: List of tuples [(split_name:str, dataset_list:list, num_samples:int, total_raw:int, num_filtered:int), ...]
        shuffle_seed: seed for shuffling the final dataset
        individual_seeds: list of seeds per split (or None to use default sequential)
        print_stats: if True, print summary statistics
    Returns:
        mixed_dataset, stats_dict
    """
    sampled_datasets = []
    stats = []

    # If not provided, generate deterministic seeds for each split
    if individual_seeds is None:
        individual_seeds = [42 + i for i in range(len(splits_and_counts))]

    for idx, (split_name, split, num_samples, total_raw, num_filtered) in enumerate(splits_and_counts):
        rng_seed = individual_seeds[idx] if idx < len(individual_seeds) else 42
        random.seed(rng_seed)

        if len(split) < num_samples:
            print(f"WARNING: Requested {num_samples} from '{split_name}', but only {len(split)} valid entries available. Using all of them.")
            sample = split
        else:
            sample = random.sample(split, num_samples)
        sampled_datasets.append(sample)
        stats.append({
            'split_name': split_name,
            'requested': num_samples,
            'actual_sampled': len(sample),
            'total_before_filter': total_raw,
            'num_filtered_out': num_filtered,
        })

    # Flatten and shuffle
    flat_dataset = [entry for sample in sampled_datasets for entry in sample]
    random.seed(shuffle_seed)
    random.shuffle(flat_dataset)

    if print_stats:
        print("\n--- Unified Dataset Statistics ---")
        for st in stats:
            print(f"Sampled from {st['split_name']}: {st['actual_sampled']}/{st['requested']} (total before filter: {st['total_before_filter']}, filtered: {st['num_filtered_out']})")
        print(f"Final unified dataset size: {len(flat_dataset)}")

        # Optionally, print some stats on output_ids distribution
        output_ids_lengths = [len(entry["output_ids"]) if "output_ids" in entry and entry["output_ids"] is not None else 0 for entry in flat_dataset]
        length_counter = Counter(output_ids_lengths)
        print("\nDistribution of output_ids lengths in unified dataset (length:count):")
        for l, cnt in sorted(length_counter.items()):
            print(f"  {l}: {cnt}")

    return flat_dataset, stats

# EXAMPLE USAGE:
# Each tuple: (label, dataset_split, num_samples_to_sample, total_raw_count, num_filtered_count)
splits_and_counts = [
    ("12.5k old cities upto 50 distractors", ds1, 12500, tot1, filt1),
    ("2.5k old cities no distractors", new_ds1, 2500, tot1, filt1),
    ("2.5k new cities 1", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 2", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 3", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 4", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 5", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 6", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 7", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 8", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 9", ds3, 2500, tot3, filt3),
    # ("2.5k new cities 10", ds3, 2500, tot3, filt3),
    # you can add more splits here, e.g., ("ds_C", ds4, 5000, tot4, filt4),
]

mixed_dataset, mix_stats = unified_sample_and_mix(splits_and_counts)

# Save the unified (mixed) dataset to a new file
mixed_save_path = "/scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_old_cities_with_up_to_50_distractors_new_cities_with_1-50_fewshots/2500_old_no_dist_12500_old_upto_50_dist_2500_new_ICL.jsonl"
mixed_save_dir = os.path.dirname(mixed_save_path)
os.makedirs(mixed_save_dir, exist_ok=True)  # Ensure the directory exists

with open(mixed_save_path, "w") as fout:
    for entry in tqdm(mixed_dataset, desc="Saving mixed dataset"):
        fout.write(json.dumps(entry) + "\n")

print(f"\nSaved mixed dataset to {mixed_save_path}")



--- Unified Dataset Statistics ---
Sampled from 150k old cities: 150000/150000 (total before filter: 19574, filtered: 0)
Sampled from 2.5k new cities 1: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 2: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 3: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 4: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 5: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 6: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 7: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 8: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 9: 2498/2500 (total before filter: 2500, filtered: 2)
Sampled from 2.5k new cities 10: 2498/2500 (total before filter: 2500, filtered: 2)
Final unified dataset size: 174980

Distributio

Saving mixed dataset: 100%|██████████| 174980/174980 [02:21<00:00, 1236.07it/s]


Saved mixed dataset to /scratch/m000122/stalaei/logs/continual_learning/data/cities_easy_synthetic_old_cities_with_up_to_50_distractors_new_cities_with_1-50_fewshots/dataset_150k_old_cities_2500_new_cities_10_splits.jsonl
